In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# 1. Convert Numpy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)



In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)



In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
for batch_images, batch_ages in train_loader:
    print(f"Batch images shape: {batch_images.shape}")
    print(f"Batch ages shape: {batch_ages.shape}")
    break

In [ ]:
# 5. Display sample images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
images, ages = next(iter(train_loader))
for i in range(5):
    img = images[i].permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f"Age: {ages[i].item():.0f}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn

class AgePredictor(nn.Module):
    def __init__(self, input_shape):
        super(AgePredictor, self).__init__()
        self.flatten_size = input_shape[0] * input_shape[1] * input_shape[2]
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    num_batches = 0
    for batch_images, batch_ages in train_loader:
        batch_images = batch_images.to(device)
        batch_ages = batch_ages.to(device)
        optimizer.zero_grad()
        predictions = model(batch_images)
        loss = criterion(predictions, batch_ages)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    return total_loss / num_batches

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0
    with torch.no_grad():
        for batch_images, batch_ages in test_loader:
            batch_images = batch_images.to(device)
            batch_ages = batch_ages.to(device)
            predictions = model(batch_images)
            loss = criterion(predictions, batch_ages)
            total_loss += loss.item()
            num_batches += 1
    return total_loss / num_batches


In [ ]:
# Task 4: Define device, model, loss, optimizer:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_shape = X_train_tensor.shape[1:]
model = AgePredictor(input_shape).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    val_loss = validate(model, test_loader, criterion, device)
    val_losses.append(val_loss)


In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses, 'b-o', label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, 'r-o', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()
images, actual_ages = next(iter(test_loader))
with torch.no_grad():
    predictions = model(images.to(device)).cpu()

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()
for i in range(8):
    img = images[i].permute(1, 2, 0).numpy()
    actual = actual_ages[i].item()
    predicted = predictions[i].item()
    axes[i].imshow(img)
    axes[i].set_title(f"Actual: {actual:.0f} | Pred: {predicted:.1f}")
    axes[i].axis('off')
plt.suptitle("Age Predictions vs Actual Ages")
plt.tight_layout()
plt.show()
